# XAI hành vi Deep SARSA UCB-VAE — BAD period only (2021–2023)

Notebook tự chứa toàn bộ mã nguồn, không gọi file `.py` phân tích bên ngoài. Trọng tâm duy nhất là sáu checkpoint UCB-VAE BAD và ba năm cuối 2021–2023.

Quy trình RDX+MSX bám theo `RDX_2013_2017.ipynb`:

1. Replay chính sách Q-greedy của checkpoint UCB-VAE BAD.
2. Phân rã Q thành Profit, Risk, Position và Stability.
3. Chọn Top-15 critical points theo thứ tự ưu tiên: đáy/max-drawdown, đảo chiều, action shift.
4. Tại mỗi critical point, so sánh action được chọn với Hold và action đối xứng (hoặc Buy/Sell mạnh nếu chọn Hold).
5. Tính RDX và MSX+ cho từng so sánh, sau đó thống kê chung trên một hình cho cả sáu cổ phiếu.

SHAP beeswarm BAD được giữ như phân tích bổ sung. File `.pth` chỉ chứa Q-network, nên cả RDX/MSX và SHAP giải thích decision core Q đã học dưới chính sách UCB-VAE.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
import hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch
from torch import nn
import shap
from scipy.signal import argrelextrema

warnings.filterwarnings("ignore")
np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device("cpu")
TICKERS = ("ACB", "FPT", "GAS", "HPG", "SSI", "VCB")
REGIMES = ("BAD",)
FEATURES = ("Close", "Balance", "Position", "MACD", "RSI", "CCI", "ADX")
COMPONENTS = ("Profit", "Risk", "Position", "Stability")
ACTIONS = np.arange(-5, 6, dtype=np.int64)
BALANCE_INIT, TRANSACTION_FEE = 1000.0, 0.001
N_SHAP_SAMPLES, N_BACKGROUND, SHAP_NSAMPLES = 60, 20, 80

def find_project_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "models" / "EIDT").exists() and (candidate / "data" / "data_storer" / "data_research").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root")

ROOT = find_project_root()
DATA_ROOT = ROOT / "data" / "data_storer" / "data_research"
FINAL_ROOT = ROOT / "application" / "EIDT_Project" / "final_result"
OUTPUT = ROOT / "application" / "FAIR_2026" / "XAI_BAD_results"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Project:", ROOT)
print("Output :", OUTPUT)
print("SHAP version:", shap.__version__)


In [ ]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7, 32), nn.ReLU(), nn.Linear(32, 11))
    def forward(self, states): return self.net(states)

@dataclass
class AgentBundle:
    ticker: str
    regime: str
    model: QNetwork
    mean: np.ndarray
    std: np.ndarray
    checkpoint_hash: str

    def q_values(self, raw_states):
        x = np.asarray(raw_states, dtype=np.float32).reshape(-1, 7)
        scaled = (x - self.mean) / self.std
        with torch.no_grad():
            return self.model(torch.as_tensor(scaled, dtype=torch.float32, device=DEVICE)).cpu().numpy()

def regime_folder(ticker, regime):
    found = [p for p in (FINAL_ROOT / ticker).iterdir() if p.is_dir() and p.name.upper() == regime]
    if len(found) != 1: raise FileNotFoundError(f"Không xác định được {ticker}/{regime}")
    return found[0]

def load_agent(ticker, regime):
    path = ROOT / "models" / "EIDT" / "UCB_BAD" / f"{ticker}_bad_sarsa_ucb_vae_best.pth"
    state_dict = torch.load(path, map_location="cpu", weights_only=True)
    model = QNetwork().to(DEVICE); model.load_state_dict(state_dict); model.eval()
    scaler = np.load(regime_folder(ticker, regime) / "frozen_scaler_train_only.npz")
    mean = np.asarray(scaler["mean"], np.float32); std = np.maximum(np.asarray(scaler["std"], np.float32), 1e-6)
    if mean.shape != (7,) or std.shape != (7,): raise ValueError(f"Scaler sai shape: {ticker}/{regime}")
    return AgentBundle(ticker, regime, model, mean, std, hashlib.sha256(path.read_bytes()).hexdigest())

def load_xai_window(ticker, regime="BAD"):
    train = pd.read_csv(DATA_ROOT / "train" / f"bad_train_{ticker}.csv")
    test = pd.read_csv(DATA_ROOT / "test" / f"bad_test_{ticker}.csv")
    frame = pd.concat([train, test], ignore_index=True)
    frame["time"] = pd.to_datetime(frame["time"], errors="raise")
    numeric = ["close", "MACD", "RSI", "CCI", "ADX"]
    frame[numeric] = frame[numeric].apply(pd.to_numeric, errors="raise")
    frame = (frame[(frame.time >= "2021-01-01") & (frame.time <= "2023-12-31")]
             .sort_values("time").drop_duplicates("time").reset_index(drop=True))
    if frame.time.iloc[0].year != 2021 or frame.time.iloc[-1].year != 2023:
        raise ValueError(f"Cửa sổ BAD 2021–2023 không đầy đủ: {ticker}")
    return frame

def raw_state(row, cash, position):
    return np.asarray([row.close, cash, position, row.MACD, row.RSI, row.CCI, row.ADX], np.float32)

def rollout(agent, data):
    cash, position = BALANCE_INIT, 0
    states, action_indices, rewards = [], [], []
    for index in range(len(data) - 1):
        row = data.iloc[index]
        state = raw_state(row, cash, position); states.append(state)
        action_index = int(agent.q_values(state)[0].argmax()); requested = int(ACTIONS[action_index])
        price = float(row.close); before = cash + position * price
        if requested > 0: executed = min(requested, int(cash // (price * (1 + TRANSACTION_FEE))))
        else: executed = -min(-requested, position)
        cash -= executed * price + abs(executed) * price * TRANSACTION_FEE; position += executed
        next_price = float(data.iloc[index + 1].close); after = cash + position * next_price
        action_indices.append(action_index); rewards.append(after - before)
    return np.asarray(states, np.float32), np.asarray(action_indices), np.asarray(rewards)


## RDX và MSX+

Phân rã “Balanced” giữ đúng logic notebook tham chiếu: Q được chuẩn hóa theo giá trị portfolio; Risk là drawdown penalty, Stability là absolute-return penalty, Position thưởng/phạt việc đi cùng xu hướng MA10; Profit là phần dư để tổng bốn thành phần bằng Q chuẩn hóa. Agent vẫn chọn action theo Q gốc. Vì Risk và Stability chỉ phụ thuộc state nên chúng giống nhau cho selected action và runner-up tại cùng thời điểm; do đó RDX tương phản của hai thành phần này bằng 0 theo đúng công thức tham chiếu.


In [ ]:
def decompose_q(agent, states, data, weights=(1.0, .5, 1.0, .1), alpha=.005):
    prices = data.close.to_numpy(float)[:len(states)]
    returns = pd.Series(prices).pct_change().fillna(0).to_numpy()
    running_max = np.maximum.accumulate(prices)
    ma10 = pd.Series(prices).rolling(10, min_periods=1).mean().to_numpy()
    q_history, selected, contrast = [], [], []
    w_profit, w_risk, w_position, w_stability = weights
    for t, state in enumerate(states):
        q = agent.q_values(state)[0]
        order = np.argsort(q); chosen, runner = int(order[-1]), int(order[-2])
        portfolio = max(float(state[1] + state[2] * prices[t]), 10.0)
        drawdown = (prices[t] - running_max[t]) / max(running_max[t], 1e-9)
        trend_sign = np.sign(prices[t] - ma10[t])
        matrix = np.zeros((11, 4), dtype=float)
        for action_index, action in enumerate(ACTIONS):
            risk = w_risk * -abs(drawdown)
            stability = w_stability * -abs(returns[t])
            if action == 0: position_value = 0.0
            elif np.sign(action) == trend_sign: position_value = w_position * alpha
            else: position_value = -w_position * alpha
            q_norm = w_profit * q[action_index] / portfolio
            profit = q_norm - (risk + position_value + stability)
            matrix[action_index] = (profit, risk, position_value, stability)
        q_history.append(matrix); selected.append(chosen); contrast.append(runner)
    return np.asarray(q_history), np.asarray(selected), np.asarray(contrast)

def analyze_msx(selected_vector, compared_vector):
    delta = np.asarray(selected_vector) - np.asarray(compared_vector)
    pros = sorted([(i, value) for i, value in enumerate(delta) if value > 0], key=lambda pair: pair[1], reverse=True)
    disadvantage = float(sum(abs(value) for value in delta if value <= 0))
    selected_components, running = [], 0.0
    for index, value in pros:
        selected_components.append(index); running += float(value)
        if running > disadvantage: break
    return delta, selected_components, running > disadvantage

def summarize_rdx_msx(agent, states, data):
    decomposed, selected, contrast = decompose_q(agent, states, data)
    deltas, membership, sufficient = [], np.zeros((len(states), 4), bool), []
    for i in range(len(states)):
        delta, components, ok = analyze_msx(decomposed[i, selected[i]], decomposed[i, contrast[i]])
        deltas.append(delta); membership[i, components] = True; sufficient.append(ok)
    return np.asarray(deltas), membership, np.asarray(sufficient), selected, contrast


def identify_critical_points(data, action_indices, action_change_threshold=3, trend_window=10, top_k=15):
    prices = data.close.to_numpy(float)[:len(action_indices)]
    actions = ACTIONS[np.asarray(action_indices, dtype=int)]
    global_min = int(np.argmin(prices))
    running_max = np.maximum.accumulate(prices)
    drawdown = (prices - running_max) / np.maximum(running_max, 1e-9)
    max_dd = int(np.argmin(drawdown))
    bottoms = list(dict.fromkeys([global_min, max_dd]))
    peaks = argrelextrema(prices, np.greater, order=trend_window)[0]
    valleys = argrelextrema(prices, np.less, order=trend_window)[0]
    reversals = np.concatenate([peaks, valleys]).astype(int)
    median = np.median(prices)
    ranked_reversals = sorted(reversals, key=lambda i: abs(prices[i] - median), reverse=True)
    differences = np.abs(np.diff(actions))
    shifts = (np.where(differences >= action_change_threshold)[0] + 1).astype(int)
    ranked_shifts = sorted(shifts, key=lambda i: differences[i - 1], reverse=True)
    selected = []
    for index in [*bottoms, *ranked_reversals, *ranked_shifts]:
        if index not in selected: selected.append(int(index))
        if len(selected) == top_k: break
    selected = sorted(selected)
    categories = {
        "Lowest/Max-DD": [i for i in selected if i in set(bottoms)],
        "Trend reversal": [i for i in selected if i in set(reversals)],
        "Action shift": [i for i in selected if i in set(shifts)],
    }
    return categories, selected

def critical_rdx_msx(agent, ticker, states, action_indices, data, top_k=15):
    q_history, _, _ = decompose_q(agent, states, data)
    categories, critical_indices = identify_critical_points(data, action_indices, top_k=top_k)
    detail_rows, explanation_rows = [], []
    for critical_rank, t in enumerate(critical_indices, start=1):
        chosen_index = int(action_indices[t]); chosen_action = int(ACTIONS[chosen_index])
        selected_vector = q_history[t, chosen_index]
        if chosen_action == 0:
            comparisons = [(10, "Strong Buy (+5)"), (0, "Strong Sell (-5)")]
        else:
            comparisons = [(5, "Hold (0)"), (int(-chosen_action + 5), f"Opposite ({-chosen_action:+d})")]
        labels = [name for name, indices in categories.items() if t in indices]
        for compared_index, compared_label in comparisons:
            compared_vector = q_history[t, compared_index]
            delta, msx_components, sufficient = analyze_msx(selected_vector, compared_vector)
            explanation_id = f"{ticker}-{critical_rank}-{compared_index}"
            disadvantage = float(sum(abs(v) for v in delta if v <= 0))
            explanation_rows.append({"explanation_id": explanation_id, "ticker": ticker,
                "critical_rank": critical_rank, "step": t, "time": data.time.iloc[t],
                "critical_type": " | ".join(labels), "selected_action": chosen_action,
                "compared_action": int(ACTIONS[compared_index]), "compared_label": compared_label,
                "msx_size": len(msx_components), "msx_sufficient": bool(sufficient),
                "disadvantage": disadvantage})
            for component_index, component in enumerate(COMPONENTS):
                detail_rows.append({"explanation_id": explanation_id, "ticker": ticker,
                    "component": component, "rdx": float(delta[component_index]),
                    "rdx_abs": float(abs(delta[component_index])),
                    "in_msx": component_index in msx_components})
    return pd.DataFrame(detail_rows), pd.DataFrame(explanation_rows), categories, critical_indices


## Kernel SHAP và beeswarm

Giống notebook tham chiếu, hàm cần giải thích là Q-value của hành động greedy, `max_a Q(s,a)`. Khác với bản cũ, mọi trạng thái đều được biến đổi bằng frozen scaler của đúng ticker/regime trước khi đưa vào mạng.


In [ ]:
def kernel_shap(agent, states, seed=42):
    rng = np.random.default_rng(seed)
    sample_index = np.linspace(0, len(states) - 1, min(N_SHAP_SAMPLES, len(states)), dtype=int)
    background_index = rng.choice(len(states), size=min(N_BACKGROUND, len(states)), replace=False)
    sample, background = states[sample_index], states[background_index]
    def predict_greedy_q(batch):
        return agent.q_values(batch).max(axis=1).astype(np.float64)
    explainer = shap.KernelExplainer(predict_greedy_q, background)
    values = np.asarray(explainer.shap_values(sample, nsamples=SHAP_NSAMPLES, silent=True), dtype=float)
    if values.shape != sample.shape: raise ValueError(f"SHAP shape {values.shape} != sample {sample.shape}")
    return sample, values, sample_index

def plot_shap_beeswarm_grid(results, regime, output_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
    for ax, ticker in zip(axes.ravel(), TICKERS):
        sample, values = results[(ticker, regime)]["shap_sample"], results[(ticker, regime)]["shap_values"]
        explanation = shap.Explanation(values=values, data=sample, feature_names=list(FEATURES))
        shap.plots.beeswarm(explanation, max_display=7, show=False, ax=ax, color_bar=False, plot_size=None, s=11)
        ax.set_title(ticker, fontweight="bold"); ax.set_xlabel("SHAP value for greedy Q")
    cmap = shap.plots.colors.red_blue
    scalar = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap=cmap); scalar.set_array([])
    colorbar = fig.colorbar(scalar, ax=axes.ravel().tolist(), fraction=.018, pad=.015)
    colorbar.set_ticks([0, 1], labels=["Low", "High"]); colorbar.set_label("Feature value")
    window = "2021–2023"
    fig.suptitle(f"Kernel SHAP Beeswarm — UCB-VAE Decision Core — {regime} ({window})", fontsize=16, fontweight="bold")
    fig.savefig(output_path, dpi=500, bbox_inches="tight", facecolor="white"); plt.show()


In [ ]:
results, rdx_frames, explanation_frames, shap_rows, audit_rows, critical_count_rows = {}, [], [], [], [], []
for ticker_index, ticker in enumerate(TICKERS):
    agent = load_agent(ticker, "BAD"); data = load_xai_window(ticker)
    states, actions, rewards = rollout(agent, data)
    rdx_detail, explanations, categories, critical_indices = critical_rdx_msx(
        agent, ticker, states, actions, data, top_k=15)
    shap_sample, shap_values, shap_index = kernel_shap(agent, states, seed=42 + ticker_index)
    results[(ticker, "BAD")] = {"agent": agent, "data": data, "states": states,
        "actions": actions, "shap_sample": shap_sample, "shap_values": shap_values}
    rdx_frames.append(rdx_detail); explanation_frames.append(explanations)
    for category, indices in categories.items():
        critical_count_rows.append({"ticker": ticker, "critical_type": category, "count": len(indices)})
    for local_i, trajectory_i in enumerate(shap_index):
        for feature_index, feature in enumerate(FEATURES):
            shap_rows.append({"ticker": ticker, "time": data.time.iloc[int(trajectory_i)],
                "feature": feature, "feature_value": float(shap_sample[local_i, feature_index]),
                "shap_value": float(shap_values[local_i, feature_index])})
    audit_rows.append({"ticker": ticker, "checkpoint_hash": agent.checkpoint_hash,
        "start_date": data.time.iloc[0].date().isoformat(), "end_date": data.time.iloc[-1].date().isoformat(),
        "trajectory_states": len(states), "critical_points": len(critical_indices),
        "rdx_msx_comparisons": len(explanations), "shap_states": len(shap_sample),
        "msx_sufficient_rate": float(explanations.msx_sufficient.mean()), "total_reward": float(rewards.sum())})
    print(f"BAD {ticker}: states={len(states)}, critical={len(critical_indices)}, "
          f"RDX/MSX={len(explanations)}, MSX sufficient={explanations.msx_sufficient.mean():.1%}")

rdx_df = pd.concat(rdx_frames, ignore_index=True)
explanations_df = pd.concat(explanation_frames, ignore_index=True)
critical_counts_df = pd.DataFrame(critical_count_rows)
shap_df, audit_df = pd.DataFrame(shap_rows), pd.DataFrame(audit_rows)
rdx_df.to_csv(OUTPUT / "rdx_at_critical_points_BAD_6stocks.csv", index=False)
explanations_df.to_csv(OUTPUT / "msx_explanations_BAD_6stocks.csv", index=False)
critical_counts_df.to_csv(OUTPUT / "critical_point_counts_BAD_6stocks.csv", index=False)
shap_df.to_csv(OUTPUT / "shap_values_BAD_6stocks.csv", index=False)
audit_df.to_csv(OUTPUT / "xai_audit_BAD_6stocks.csv", index=False)
audit_df


In [ ]:
# Hình cuối cùng: thống kê RDX -> MSX trên Top-15 critical points của sáu cổ phiếu BAD.
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.subplots_adjust(top=.88, bottom=.12, hspace=.34, wspace=.24)

# (A) RDX mean absolute component difference.
rdx_matrix = (rdx_df.groupby(["component", "ticker"]).rdx_abs.mean().unstack("ticker")
              .loc[list(COMPONENTS), list(TICKERS)])
image = axes[0, 0].imshow(rdx_matrix.values, aspect="auto", cmap="viridis")
axes[0, 0].set_xticks(range(6), TICKERS); axes[0, 0].set_yticks(range(4), COMPONENTS)
axes[0, 0].set_title("(A) RDX mean |Δ component|")
fig.colorbar(image, ax=axes[0, 0], shrink=.82, label="Mean absolute RDX")

# (B) MSX component inclusion frequency.
membership = (rdx_df.groupby(["ticker", "component"]).in_msx.mean().unstack("component")
              .loc[list(TICKERS), list(COMPONENTS)])
x = np.arange(len(TICKERS)); width = .19
component_colors = {"Profit":"#2369BD", "Risk":"#D14A61", "Position":"#E6862D", "Stability":"#4E9F6D"}
for j, component in enumerate(COMPONENTS):
    axes[0, 1].bar(x + (j - 1.5) * width, membership[component], width,
                   label=component, color=component_colors[component])
axes[0, 1].set_xticks(x, TICKERS); axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].set_ylabel("Fraction of explanations"); axes[0, 1].set_title("(B) MSX+ inclusion frequency")
axes[0, 1].grid(axis="y", alpha=.25); axes[0, 1].legend(ncol=2, frameon=False, fontsize=8)

# (C) MSX explanation size.
size_stats = explanations_df.groupby("ticker").msx_size.agg(["mean", "std"]).loc[list(TICKERS)]
axes[1, 0].bar(TICKERS, size_stats["mean"], yerr=size_stats["std"].fillna(0), capsize=4, color="#6C5B7B")
for i, value in enumerate(size_stats["mean"]): axes[1, 0].text(i, value + .04, f"{value:.2f}", ha="center", fontsize=8)
axes[1, 0].set_ylabel("Number of components"); axes[1, 0].set_title("(C) MSX+ size — mean ± SD")
axes[1, 0].grid(axis="y", alpha=.25)

# (D) Composition of selected critical points (multi-label counts).
count_matrix = (critical_counts_df.pivot(index="ticker", columns="critical_type", values="count")
                .fillna(0).loc[list(TICKERS)])
bottom = np.zeros(len(TICKERS)); category_colors = ("#C44E52", "#4C72B0", "#55A868")
for color, category in zip(category_colors, ["Lowest/Max-DD", "Trend reversal", "Action shift"]):
    values = count_matrix.get(category, pd.Series(0, index=TICKERS)).to_numpy()
    axes[1, 1].bar(TICKERS, values, bottom=bottom, label=category, color=color); bottom += values
axes[1, 1].set_ylabel("Multi-label count among Top-15")
axes[1, 1].set_title("(D) Critical-point composition")
axes[1, 1].legend(frameon=False, fontsize=8); axes[1, 1].grid(axis="y", alpha=.25)

fig.suptitle("RDX-derived MSX Statistics — UCB-VAE Agents — BAD Period (2021–2023)",
             fontsize=16, fontweight="bold", y=.97)
fig.text(.5, .035, "Per stock: Top-15 critical points × 2 contrastive actions = 30 RDX/MSX explanations.",
         ha="center", fontsize=9, color="#444")
png = OUTPUT / "RDX_MSX_statistics_BAD_6stocks.png"
pdf = OUTPUT / "RDX_MSX_statistics_BAD_6stocks.pdf"
fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(pdf, bbox_inches="tight", facecolor="white")
plt.show(); plt.close(fig)

plot_shap_beeswarm_grid(results, "BAD", OUTPUT / "SHAP_beeswarm_BAD_6stocks_2021_2023.png")
print("Saved BAD-only RDX/MSX statistics and SHAP outputs to:", OUTPUT)


## Kết quả BAD-only

Hình `RDX_MSX_statistics_BAD_6stocks.png` là hình chính để phân tích và so sánh sáu agent. Mỗi cổ phiếu dùng đúng 15 critical points và hai contrastive comparisons tại mỗi điểm, tổng cộng 30 giải thích RDX/MSX.

![RDX MSX BAD](XAI_BAD_results/RDX_MSX_statistics_BAD_6stocks.png)

### SHAP bổ sung

![SHAP BAD](XAI_BAD_results/SHAP_beeswarm_BAD_6stocks_2021_2023.png)
